In [1]:
import pandas as pd
import numpy as np
import xarray as xr 

import sys
import datetime
import re
from datetime import date

In [2]:
### Define date to append to output file names
today = date.today()
run_date = today.strftime("%Y%m%d")

In [3]:
# load in master RCA instrument list
master_df = pd.read_csv('./params/RCA-InstrumentList.csv')

In [4]:
master_df

,assetID,instrumentType,mfgSN,SNnotes
0,ATOSU-69825-00001,ADCPS-I,21498,NaN
1,ATOSU-69825-00002,ADCPS-I,18153,NaN
2,ATOSU-69825-00003,ADCPS-I,18919,NaN
3,ATAPL-58419-00001,ADCPS-K,18444,NaN
4,ATAPL-58419-00002,ADCPS-K,18975,NaN
...,...,...,...,...
399,ATAPL-71444-00003,DP-VEHICLE,3,NaN
400,ATAPL-71444-00004,DP-VEHICLE,4,NaN
401,ATAPL-71444-00005,DP-VEHICLE,5,NaN
402,ATAPL-71444-00006,DP-VEHICLE,6,NaN


In [5]:
# load manual image overview CSV for the current year
imageSN_current = pd.read_csv('./inputs/imageSN_2024_draft.csv')
imageSN_past = pd.read_csv('./inputs/imageSN_2021_draft.csv')

In [6]:
#imageSN = pd.concat([imageSN_current, imageSN_past]) #TODO
imageSN = imageSN_current # unless you are loading in multiple years of image verification csvs

In [7]:
imageSN

,referenceDesignator,deployYear,imageFile,imageSerialNumber,imageAssetID
0,CE02SHBP-LJ01D-05-ADCPTB104,2020,IMG_4469.HEIC,19003,ATOSU-69826-00002
1,CE02SHBP-LJ01D-06-CTDBPN106,2020,IMG_4466.HEIC,7230,ATOSU-69827-00001
2,CE02SHBP-LJ01D-06-DOSTAD106,2020,IMG_4467.HEIC,216,ATOSU-58320-00020
3,CE02SHBP-LJ01D-07-VEL3DC108,2020,IMG_4468.HEIC,"5007,8167",ATOSU-69829-00003
4,CE02SHBP-LJ01D-08-OPTAAD106,2020,IMG_4465.HEIC,169,ATOSU-58332-00009
...,...,...,...,...,...
64,RS03AXPS-SF03A-3B-OPTAAD301,2020,IMG_1546.JPG,134,ATAPL-58332-00001
65,RS03AXPS-SF03A-3D-SPKIRA301,2020,IMG_1548.JPG,244,ATAPL-58341-00002
66,RS03AXPS-SF03A-4A-NUTNRA301,2020,IMG_1544.JPG,379,ATAPL-68020-00002
67,RS03AXPS-SF03A-4B-VELPTD302,2020,IMG_1544.JPG,6334,ATAPL-70114-00002


In [8]:
def process_mfgSN(row):
    # some assests have multiple manufacturer serial numbers so we split them out
    mfgSN_values = row['mfgSN'].split(',')
    assetID = row['assetID']
    
    return {mfgSN_value: assetID for mfgSN_value in mfgSN_values}

# Applying the function along rows (axis=1)
mfg_dict_list = master_df.apply(process_mfgSN, axis=1).to_list()

In [9]:
instrument_AT_dict = master_df.set_index('assetID')['instrumentType'].to_dict()
fuzzy_instrument_dict = {key: re.sub(r'[^a-zA-Z0-9]', '', value[:5]) for key, value in instrument_AT_dict.items()} #strip out anything thats not a letter or a number the "fuzzy" matching falls apart

In [10]:
fuzzy_instrument_dict

{'ATOSU-69825-00001': 'ADCPS',
 'ATOSU-69825-00002': 'ADCPS',
 'ATOSU-69825-00003': 'ADCPS',
 'ATAPL-58419-00001': 'ADCPS',
 'ATAPL-58419-00002': 'ADCPS',
 'ATAPL-58419-00003': 'ADCPS',
 'ATOSU-69826-00001': 'ADCPT',
 'ATOSU-69826-00002': 'ADCPT',
 'ATOSU-69826-00003': 'ADCPT',
 'ATAPL-58315-00001': 'ADCPT',
 'ATAPL-58315-00002': 'ADCPT',
 'ATAPL-58315-00003': 'ADCPT',
 'ATAPL-58315-00004': 'ADCPT',
 'ATAPL-58315-00005': 'ADCPT',
 'ATAPL-68073-00001': 'ADCPT',
 'ATAPL-68073-00002': 'ADCPT',
 'ATAPL-68073-00003': 'ADCPT',
 'ATAPL-68073-00004': 'ADCPT',
 'ATAPL-68073-00005': 'ADCPT',
 'PIRSN-PADCPA-00001': 'ADCPF',
 'ATAPL-58316-00001': 'BOTPT',
 'ATAPL-58316-00002': 'BOTPT',
 'ATAPL-58316-00003': 'BOTPT',
 'ATAPL-58316-00004': 'BOTPT',
 'ATAPL-58316-00005': 'BOTPT',
 'ATAPL-78437-00001': 'CAMDS',
 'ATAPL-78437-00002': 'CAMDS',
 'ATAPL-78437-00003': 'CAMDS',
 'ATAPL-78437-00004': 'CAMDS',
 'ATAPL-78437-00005': 'CAMDS',
 'ATAPL-78437-00006': 'CAMDS',
 'ATAPL-78437-00007': 'CAMDS',
 'ATAPL

In [11]:
mfg_dict_list[:20]

[{'21498': 'ATOSU-69825-00001'},
 {'18153': 'ATOSU-69825-00002'},
 {'18919': 'ATOSU-69825-00003'},
 {'18444': 'ATAPL-58419-00001'},
 {'18975': 'ATAPL-58419-00002'},
 {'18977': 'ATAPL-58419-00003'},
 {'18493': 'ATOSU-69826-00001'},
 {'19003': 'ATOSU-69826-00002'},
 {'22115': 'ATOSU-69826-00003'},
 {'18478': 'ATAPL-58315-00001'},
 {'18974': 'ATAPL-58315-00002'},
 {'18980': 'ATAPL-58315-00003'},
 {'23338': 'ATAPL-58315-00004'},
 {'23339': 'ATAPL-58315-00005'},
 {'18471': 'ATAPL-68073-00001'},
 {'18813': 'ATAPL-68073-00002'},
 {'19224': 'ATAPL-68073-00003'},
 {'23442': 'ATAPL-68073-00004'},
 {'23443': 'ATAPL-68073-00005'},
 {'1023': 'PIRSN-PADCPA-00001'}]

In [12]:
def instrument_type_sanity_check(instrument, possible_asset_tag, inst_dict):
    instrument_type = inst_dict[possible_asset_tag]

    if instrument_type in instrument:
        print(f"{instrument_type} for possible AT {possible_asset_tag} matches {instrument}")
        return True
    else: 
        print(f"{instrument_type} for possible AT {possible_asset_tag} DOES NOT match {instrument}")
        return False

In [13]:
def process_imageSN(row, mfg_dict_list, inst_dict):
    
    print('-----------------------------------------------')
    imageSerialNumbers = row['imageSerialNumber']
    imageAssetID = row['imageAssetID']
    instrument = row['referenceDesignator'][18:]
    print(instrument)
    
    if type(imageSerialNumbers) is str and '/' in imageSerialNumbers:
        imageSerialNumbers = row['imageSerialNumber'].split('/')
        print(f"it looks like there are multiple / SNs in this cell {imageSerialNumbers}")
    elif type(imageSerialNumbers) is str and ',' in imageSerialNumbers:
        imageSerialNumbers = row['imageSerialNumber'].split(',')
        print(f"it looks like there are multiple , SNs in this cell {imageSerialNumbers}")
    else:
        imageSerialNumbers = [imageSerialNumbers] # both list for consistency

    # list of matching asset ids
    match_list = []
    # proportion of serial number which matches if it is a partial match
    prop_list = []
    # list of the manufacturer serial numbers that match the imageSN
    mfg_sn_list = []

    # exact SN match
    exact_match_found = False 
    # assetID, exact SN or partial SN match
    any_match_found = False
    # exact assetID match
    exact_asset_match_found = False
    # any assetID match
    any_asset_match_found = False

    # extract lists of mfg serial numbers and asset tags from mfg_dict_list
    
    for imageSerialNumber in imageSerialNumbers:
        print(f"matching {imageSerialNumber}...")
        for SN_AT_dict in mfg_dict_list:
            for manufacturerSerialNumber in SN_AT_dict.keys():
                if type(imageSerialNumber) is str and imageSerialNumber == manufacturerSerialNumber:
                    matching_assetID = SN_AT_dict[manufacturerSerialNumber]
                    if instrument_type_sanity_check(instrument, matching_assetID, inst_dict):
                        print("exact match found and sanity check passed")
                        match_list.append(matching_assetID)
                        mfg_sn_list.append(manufacturerSerialNumber)
    
                        match_list = list(set(match_list))
        
                        exact_match_found = True
                        any_match_found = True
            
        if exact_match_found == False:
            for SN_AT_dict in mfg_dict_list:
                for manufacturerSerialNumber in SN_AT_dict.keys():
                    if type(imageSerialNumber) is str and imageSerialNumber in manufacturerSerialNumber and imageSerialNumber != manufacturerSerialNumber:
                        matching_assetID = SN_AT_dict[manufacturerSerialNumber]
                        if instrument_type_sanity_check(instrument, matching_assetID, inst_dict):
                            print("found a partial match and sanity check passed")
                            match_prop = round(len(imageSerialNumber) / len(manufacturerSerialNumber),2)
                            match_list.append(matching_assetID)
                            prop_list.append(match_prop)
                            mfg_sn_list.append(manufacturerSerialNumber)
        
                            match_list = list(set(match_list))
            
                            any_match_found = True

        
        if any_match_found == False:
            print(f"still no match - attempting to match by assetID <{imageAssetID}> instead...")
            for SN_AT_dict in mfg_dict_list:
                for master_assetID in SN_AT_dict.values():
                    if type(imageAssetID) is str and imageAssetID == master_assetID:
                        if instrument_type_sanity_check(instrument, imageAssetID, inst_dict):
                            print("exact assetID match found from image SN, and sanity check passed")
                            matching_assetID = master_assetID
                            match_list.append(matching_assetID)
        
                            match_list = list(set(match_list))
            
                            any_match_found = True
                            exact_asset_match_found = True
                
            
    if len(match_list) > 0:
        row['matching_asset_ids'] = match_list
    else: 
        row['matching_asset_ids'] = np.nan

    if len(prop_list) > 0: 
        row['proportion_match'] = prop_list
    else: 
        row['proportion_match'] = np.nan

    if len(mfg_sn_list) > 0:
        row['matching_mfg_sn'] = mfg_sn_list
    else:
        row['matching_mfg_sn'] = np.nan

    if exact_match_found == False:
        row["exact_SN_match"] = np.nan
    else:
        row["exact_SN_match"] = True

    if exact_asset_match_found == False:
        row["exact_assetID_match"] = np.nan
    else:
        row["exact_assetID_match"] = True

    if any_match_found == False:
        row["any_match"] = np.nan
    else:
        row["any_match"] = True

    return row



In [14]:
log_file_path = 'logs/fuzzy_match_log.txt'
with open(log_file_path, 'w') as log_file:
    # Save the current stdout so we can restore it later
    original_stdout = sys.stdout
    sys.stdout = log_file
    
    imageSN = imageSN.apply(process_imageSN, axis=1, args=(mfg_dict_list, fuzzy_instrument_dict))

    sys.stdout = original_stdout
print("logs printed to logs directory")

logs printed to logs directory


In [15]:
imageSN.head()

,referenceDesignator,deployYear,imageFile,imageSerialNumber,imageAssetID,matching_asset_ids,proportion_match,matching_mfg_sn,exact_SN_match,exact_assetID_match,any_match
0,CE02SHBP-LJ01D-05-ADCPTB104,2020,IMG_4469.HEIC,19003,ATOSU-69826-00002,[ATOSU-69826-00002],NaN,[19003],True,NaN,True
1,CE02SHBP-LJ01D-06-CTDBPN106,2020,IMG_4466.HEIC,7230,ATOSU-69827-00001,[ATOSU-69827-00001],[0.31],[16P71176-7230],NaN,NaN,True
2,CE02SHBP-LJ01D-06-DOSTAD106,2020,IMG_4467.HEIC,216,ATOSU-58320-00020,[ATOSU-58320-00020],NaN,[216],True,NaN,True
3,CE02SHBP-LJ01D-07-VEL3DC108,2020,IMG_4468.HEIC,"5007,8167",ATOSU-69829-00003,[ATOSU-69829-00003],"[0.5, 0.44]","[VEC-5007, VEC-8167]",NaN,NaN,True
4,CE02SHBP-LJ01D-08-OPTAAD106,2020,IMG_4465.HEIC,169,ATOSU-58332-00009,[ATOSU-58332-00009],NaN,[169],True,NaN,True


In [16]:
imageSN.to_csv(f"./reportOuts/fuzzyMatches_{run_date}_2020.csv")
imageSN_HITL = imageSN
imageSN_HITL['HITL_match_notes'] = pd.Series(dtype='object')
imageSN_HITL.to_csv(f"./reportOuts/fuzzyMatches_HITL_{run_date}_2020.csv")

## NOTE: Now we have to manually go through the fuzzy matches HITL csv and correct multiply matches, account for missing AssetIDs and serial numbers.